In [2]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from IPython.display import  Markdown
from dotenv.ipython import load_dotenv

In [3]:
load_dotenv(override=True)

True

In [4]:
llm=ChatOllama(model="qwen3.5:9b", temperature=0)

In [5]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant.",
)

In [6]:
resp=agent.invoke(input={"messages": [
    {"role": "user", "content": "What is the capital of France?"}
]}
   
)

In [7]:
print(resp['messages'][-1].content)

The capital of France is **Paris**.


In [8]:
resp=agent.invoke(input={"messages": [
    {"role": "user", "content": "l'historique de notre discussion?"}
]}
)

In [9]:
print(resp['messages'][-1].content)

Je ne peux pas accéder à l'historique de nos discussions précédentes, car chaque session est indépendante. Cependant, je suis là pour vous aider ! N'hésitez pas à me poser une nouvelle question ou à me donner des détails sur ce dont vous avez besoin. 😊


In [10]:
from langchain.agents.middleware import ModelRequest , ModelResponse , wrap_model_call
from langchain.messages import SystemMessage , HumanMessage , AIMessage

In [11]:
llm=ChatOllama(model="qwen3.5:9b", temperature=0)
llm2=ChatOllama(model="qwen3.5:2b", temperature=0)

In [12]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest , handler)->ModelResponse:
    env = request.runtime.context.get("env", "production")
    if env == "production":
        model = llm
        print("Using llm 9b")
    else:
        model = llm2
        print("Using llm 2b")
    return handler(request.override(model=model))

In [13]:
agent2 = create_agent(
    model=llm,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

In [14]:
from IPython.display import Markdown

In [15]:
resp=agent2.invoke(input={"messages": [HumanMessage(content="What is the capital of France?")]},
              context={"env": "production"}
              )

[values] {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='35453b37-6323-44b3-8682-d62b200b293f')]}
Using llm 9b
[updates] {'model': {'messages': [AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-08T17:42:07.4809902Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26015124700, 'load_duration': 219526700, 'prompt_eval_count': 17, 'prompt_eval_duration': 795150500, 'eval_count': 121, 'eval_duration': 24912078600, 'logprobs': None, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d6e2f-85d8-7a11-8fbe-9f4d11cd8d09-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 121, 'total_tokens': 138})]}}
[values] {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='35453b37-6323-44b3-8682-d62b200b293f'),

In [16]:
print(display(Markdown(resp['messages'][-1].content)))

The capital of France is **Paris**.

None


In [17]:
from langgraph.checkpoint.memory import InMemorySaver

In [18]:
memory=InMemorySaver()
agent = create_agent(
    model=ChatOllama(model="qwen3.5:2b", temperature=0),
    system_prompt="You are a helpful assistant.",
    checkpointer=memory
)

In [19]:
config = {"configurable": {"thread_id": 1}}
resp=agent.invoke(
    input={"messages": [HumanMessage(content="My name's Reyane")]},
    config=config
)

In [20]:
print(display(Markdown(resp['messages'][-1].content)))

Hello Reyane! It's nice to meet you. How can I help you today?

None


In [21]:
resp=agent.invoke(input={"messages": [HumanMessage(content="what's my name?")]},
                  config=config)

In [22]:
print(display(Markdown(resp['messages'][-1].content)))

Yes, your name is Reyane. How can I help you today?

None


In [23]:
from langchain.tools import tool

In [24]:
@tool
def get_meteo(city:str) -> str:
    """Get the current weather for a city."""
    print("weather tool invoked")
    return {"city": city, "temperature": "20°C", "condition": "Sunny"} 
@tool
def get_employee_info(employee_name: str) -> str:
    """Get information about a specific employee."""
    print("employee info tool invoked")
    return {"name": employee_name, "salary": "$75,000", "seniority": "5 years"} 

In [25]:
agent4 = create_agent(
    model=ChatOllama(model="qwen3.5:2b", temperature=0),
    tools=[get_meteo, get_employee_info],
    checkpointer=memory,
    system_prompt="Answer user questions using provided tools"
)

In [26]:
resp = agent4.invoke(input={"messages": [HumanMessage(content="What's the weather in Paris?")]},
              config={"configurable": {"thread_id": 1}})

weather tool invoked


In [27]:
print(display(Markdown(resp['messages'][-1].content)))

The weather in Paris is currently **20°C** and it is **sunny**.

None


In [28]:
resp = agent4.invoke(input={"messages": [HumanMessage(content="Give me all the details you have about the employee simo")]},
              config={"configurable": {"thread_id": 1}})

employee info tool invoked


In [29]:
print(display(Markdown(resp['messages'][-1].content)))

Here are the details for the employee **simo**:

*   **Name:** simo
*   **Salary:** $75,000
*   **Seniority:** 5 years

None


In [30]:
load_dotenv(override=True)

True

In [32]:
from langchain_tavily import TavilySearch

In [33]:
tavily = TavilySearch(max_results=3 , search_depth="advanced")

In [40]:
@tool
def search_web(query: str) -> str:
    """Search the web for a query."""
    print("web search tool invoked")
    Results = tavily.invoke({"query": query})
    return Results

In [41]:

agent5 = create_agent(
    model=ChatOllama(model="qwen3.5:2b", temperature=0),
    tools=[get_meteo, get_employee_info, search_web],
    checkpointer=memory,
    system_prompt="Answer user questions using provided tools"
)

In [48]:
resp=agent5.invoke(input={"messages": [HumanMessage(content="search in the web the weather in New York?")]},
              config=config)

web search tool invoked


In [49]:
print(display(Markdown(resp['messages'][-1].content)))

I found several weather sources for New York. Here's the information:

## Weather in New York

### April 2026 Forecast (from multiple sources):

**Temperature:**
- Average daily temperature: 16°C (61°F)
- Range: 13°C (55°F) to 24°C (75°F)

**Precipitation:**
- Average rainfall: 15.9mm (0.63 inches)
- Rainy days: 8 days
- Dry days: 21 days

**Humidity:**
- Average humidity: 72%
- Range: 71% to 75%

**Weather Conditions:**
- Mostly cloudy to partly cloudy
- Some light rain showers possible
- Humidity between 71% and 75%

### Monthly Overview:
- **January:** -1°C to -4°C, 66% humidity, 7 rainy days
- **February:** 0°C to -3°C, 64% humidity, 6 rainy days
- **March:** 4°C to 0°C, 65% humidity, 7 rainy days
- **April:** 10°C to 6°C, 65% humidity, 8 rainy days
- **May:** 16°C to 12°C, 68% humidity, 8 rainy days
- **June:** 21°C to 17°C, 70% humidity, 8 rainy days
- **July:** 24°C to 20°C, 66% humidity, 8 rainy days
- **August:** 23°C to 19°C, 69% humidity, 8 rainy days
- **September:** 24°C to 17°C, 66% humidity, 6 rainy days

The weather in New York is generally mild with moderate humidity and occasional rainfall throughout the year.

None


In [59]:
from langchain_experimental.tools import PythonREPLTool

In [63]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [66]:
agent6 = create_agent(
    model=ChatOllama(model="qwen3.5:2b", temperature=0),
    tools=[repl_tool],
    system_prompt="Génère et exécute du code python en plaçant le résultat et le code dans le fichier doc.txt",
    debug=True
)

In [65]:
resp=agent6.invoke(input={"messages": [HumanMessage(content="Génère un code python qui affiche 'Hello World' et place le résultat et le code dans le fichier doc.txt")]},
)

In [67]:
print(display(Markdown(resp['messages'][-1].content)))

Le fichier `doc.txt` a été généré avec succès. Voici le contenu :

```
Résultat:
Hello World

Code Python:
print("Hello World")
```

None
